# MÅL

Bruke reddits webapi til å:
- gjøre et søk på "elbiler"
- Hente alle artikler ("treff") fra det siste året
- Hente ut alle (nesten) kommentarer til innleggene
- putte det i pandas dataframe

evt:
- Gjøre sentimentanalyse med chatgpt api positiv/nøytral/negativ
- plotte utvikling (om noen)

In [5]:
import requests, json
import pandas as pd


In [6]:
with open("reddit_tokens.json", "r") as file:
    tokens = json.load(file)

access_token = tokens["access_token"]
refresh_token = tokens["refresh_token"]

client_id = "v2uZeXUHIszhF2K4hNOksQ"
client_secret = "2qamP2_KAEG7eNkXMwrJPhbb6jxzKw"

def refresh_tokens():
    global access_token, refresh_token
    refresh_url = "https://www.reddit.com/api/v1/access_token"
    payload = {"grant_type": "refresh_token", "refresh_token": refresh_token}
    headers = {"User-Agent": "python:undervisning_h25"}
    res = requests.post(refresh_url, auth=(client_id, client_secret), data=payload, headers=headers)
    res.raise_for_status()
    tokens = res.json()
    access_token = tokens["access_token"]
    refresh_token = tokens["refresh_token"]
    with open("reddit_tokens.json", "w") as file:
        json.dump(tokens,file)


def reddit_get(endpoint, params=None):
    base_url = "https://oauth.reddit.com"
    headers = {"Authorization": f"Bearer {access_token}",
              "User-Agent": "python:undervisning_h25"}
    res = requests.get(base_url+endpoint, headers=headers, params=params)
    res.raise_for_status()
    return res.json()

def reddit_search(q):
    params =  {
        "q": q,
        "sort": "top",
        "t": "year",
        "limit": 100
    }
    res_sider = []
    res = reddit_get("/search", params)
    res_sider.append(res)
    after = res["data"]["after"]
    
    while after:
        params = {
            "q": q,
            "sort": "top",
            "t": "year",
            "limit": 100,
            "after": after
        }
        res = reddit_get("/search", params)
        res_sider.append(res)
        after = res["data"]["after"]
    return res_sider



In [7]:
params = {
    "q": "elbil",
    "sort": "top",
    "t": "year",
    "limit": 100
}

treff = reddit_search("elbil")



In [8]:
dfs = [ pd.json_normalize(dat, record_path=["data","children"]) for dat in treff]
df = pd.concat(dfs)
kolonner = ["kind", "data.subreddit", "data.selftext", 
            "data.author_fullname", "data.title", 
            "data.name", "data.id", "data.created", "data.url"]
df_search = df[kolonner]
df_search = df_search.set_index(pd.PeriodIndex(pd.to_datetime(df_search["data.created"],unit="s", utc=True), freq="D"))
df_search = df_search.sort_index()
df_search

,kind,data.subreddit,data.selftext,data.author_fullname,data.title,data.name,data.id,data.created,data.url
data.created,,,,,,,,,
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...
2024-12-22,t3,elbilsverige,Helt sen jag investerade i en Elbil har jag gå...,t2_fea2scsox,"Elbil, en trevlig resa",t3_1hk19x3,1hk19x3,1.734883e+09,https://www.reddit.com/r/elbilsverige/comments...
2024-12-22,t3,Salamanders40k,Hello everyone. I Am brand new to Warhammer 40...,t2_8w0u6qkh7,New guy to Warhammer.,t3_1hk1oau,1hk1oau,1.734884e+09,https://i.redd.it/ogid7eakdf8e1.jpeg
2024-12-26,t3,elbilsverige,Ska strax påbörja andra resan till fjällen med...,t2_d7idzowy,Andra fjällresan,t3_1hmm9ed,1hmm9ed,1.735211e+09,https://www.reddit.com/r/elbilsverige/comments...
2024-12-27,t3,sweden,Finns alltid några procent extra kvar under hu...,t2_d7idzowy,Den bästa anledningen till att elbil är bättre...,t3_1hncxk3,1hncxk3,1.735299e+09,https://i.redd.it/6v9tadh5md9e1.jpeg
...,...,...,...,...,...,...,...,...,...
2025-12-13,t3,dkbiler,"Jeg har ventet spændt på, at BMW får lanceret ...",t2_37utfpb2,Er der fremtid i elbiler i Danmark når den ful...,t3_1plgpyu,1plgpyu,1.765615e+09,https://www.reddit.com/r/dkbiler/comments/1plg...
2025-12-14,t3,HTML,,t2_1rwjy16cos,How would I make a website like Arngren.net?,t3_1pmmuoe,1pmmuoe,1.765741e+09,https://i.redd.it/h464paxl287g1.png
2025-12-14,t3,elbilsverige,Jag kör en Renault Zoe utan CCS sedan i somras...,t2_7yj2btor,Laddar ni vid butiker?,t3_1pmalpv,1pmalpv,1.765707e+09,https://www.reddit.com/r/elbilsverige/comments...


In [9]:
treff["data"]["children"][0]["data"]["selftext"]
#print("Antall treff", len(treff["data"]["children"]))
#for post in treff["data"]["children"]:
#    print(post["data"]["selftext"])
#    print("------------------------------\n\n")

TypeError: list indices must be integers or slices, not str

In [10]:
kommentar[2]["data"]

NameError: name 'kommentar' is not defined

In [12]:

testid = df_search.iloc[0,-3]
url = df_search.iloc[0,-1]

comments_url = "/comments/article"
params = {"article": testid}

kommentar = reddit_get(comments_url, params)

with open("kommentartest.json", "w") as file:
    json.dump(kommentar,file)



In [13]:
kommentarer = kommentar[1]["data"]["children"].copy()
out = []
while len(kommentarer) > 0:
    kom = kommentarer.pop()
    if kom["kind"] == "Listing":
        kommentarer.extend(kom["data"]["children"])
    else:
        out.append(kom["data"]["body"])
        if isinstance(kom["data"]["replies"], dict):
            kommentarer.append(kom["data"]["replies"])



In [14]:
def get_comments(artikkel_id):
    comments_url = "/comments/article"
    params = {"article": artikkel_id}
    data = reddit_get(comments_url, params)
    kommentarer = data[1]["data"]["children"].copy()
    out = []
    while len(kommentarer) > 0:
        kom = kommentarer.pop()
        if kom["kind"] == "Listing":
            kommentarer.extend(kom["data"]["children"])
        elif kom["kind"] == "t1":
            out.append(kom["data"]["body"])
            if isinstance(kom["data"]["replies"], dict):
                kommentarer.append(kom["data"]["replies"])
    return out

testid = "1nqsvr8"
kommentarer = get_comments(testid)


In [15]:
%%time
df_search["kommentarer"] = df_search["data.id"].map(get_comments)


CPU times: user 1.06 s, sys: 103 ms, total: 1.17 s
Wall time: 2min 43s


In [16]:
df_search.set_index("data.id")
n_comments = 243

n_comments += df_search["kommentarer"].map(len).sum()
print("antall innlegg + kommentarer = ", n_comments)

antall innlegg + kommentarer =  15557


In [17]:
df_search.index.name = "dato"
df = df_search.copy()

In [18]:
df = df.explode("kommentarer")
df

,kind,data.subreddit,data.selftext,data.author_fullname,data.title,data.name,data.id,data.created,data.url,kommentarer
dato,,,,,,,,,,
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,De første mange afbetalinger på dit billån er ...
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,Jeg vil bare hilse fra der skred fra Nordea i ...
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,Jeg har den opfattelse er at den friværdi man ...
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,Sælg bilen og køb en I har råd til istedet. Hv...
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,Hvem siger de ikke har råd til den? \n\nOP har...
...,...,...,...,...,...,...,...,...,...,...
2025-12-18,t3,elbilsverige,Jag har haft en diesel-driven *Nissan Qashqai ...,t2_1lrih27f,"Ska snart hämta ut en Kia EV6 GT Line, börjar ...",t3_1pplall,1pplall,1.766045e+09,https://www.reddit.com/r/elbilsverige/comments...,"Jo, men jag föredrar att om jag ska köra på en..."
2025-12-18,t3,elbilsverige,Jag har haft en diesel-driven *Nissan Qashqai ...,t2_1lrih27f,"Ska snart hämta ut en Kia EV6 GT Line, börjar ...",t3_1pplall,1pplall,1.766045e+09,https://www.reddit.com/r/elbilsverige/comments...,Det där brukar du kunna ställa in själv. Därem...
2025-12-18,t3,elbilsverige,Jag har haft en diesel-driven *Nissan Qashqai ...,t2_1lrih27f,"Ska snart hämta ut en Kia EV6 GT Line, börjar ...",t3_1pplall,1pplall,1.766045e+09,https://www.reddit.com/r/elbilsverige/comments...,"Tack, det här var nog faktiskt precis vad jag ..."


In [247]:
%%time
a = 0
for i in range(1000):
    a += i
print("noe greier")

noe greier
CPU times: user 586 µs, sys: 0 ns, total: 586 µs
Wall time: 597 µs


In [15]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from openai import OpenAI
import os

# 1. The inner-most model
class User(BaseModel):
    name: str = Field(description="The full name of the person")
    role: str = Field(description="Their job title or department")

# 2. The middle-layer model
class Task(BaseModel):
    title: str
    priority: Literal["high", "medium", "low"]
    assignee: User  # Nesting the User model here

# 3. The root model
class Project(BaseModel):
    project_name: str
    deadline: str
    tasks: List[Task] # A list of nested Task objects
    budget_approved: bool

Project.model_json_schema()

{'$defs': {'Task': {'properties': {'title': {'title': 'Title',
     'type': 'string'},
    'priority': {'enum': ['high', 'medium', 'low'],
     'title': 'Priority',
     'type': 'string'},
    'assignee': {'$ref': '#/$defs/User'}},
   'required': ['title', 'priority', 'assignee'],
   'title': 'Task',
   'type': 'object'},
  'User': {'properties': {'name': {'description': 'The full name of the person',
     'title': 'Name',
     'type': 'string'},
    'role': {'description': 'Their job title or department',
     'title': 'Role',
     'type': 'string'}},
   'required': ['name', 'role'],
   'title': 'User',
   'type': 'object'}},
 'properties': {'project_name': {'title': 'Project Name', 'type': 'string'},
  'deadline': {'title': 'Deadline', 'type': 'string'},
  'tasks': {'items': {'$ref': '#/$defs/Task'},
   'title': 'Tasks',
   'type': 'array'},
  'budget_approved': {'title': 'Budget Approved', 'type': 'boolean'}},
 'required': ['project_name', 'deadline', 'tasks', 'budget_approved'],
 '

In [132]:
from pydantic import BaseModel, Field
from openai import OpenAI
import os, time


class analyse_kommentar(BaseModel):
    nummer: int = Field("kommentarnummer")
    analyse: int = Field("Sentimentanalyse av kommentar #nummer# (0 nøytral, -1 negativ, +1 positiv")

class sentimentanalyse(BaseModel):
    post: int = Field("sentimentanalyse av hovedartikkel: 0 nøytral/ikke-relevant, -1 negativ, +1 positiv")
    kommentar: list[analyse_kommentar]


client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def analyser(komm):
    time.sleep(1)
    
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Du er ekspert på sentimentanalyse av holdninger rundt elbiler og det grønne skiftet og ser på lister med innlegg og kommentarer fra reddit på norsk, dansk og svensk og vurderer om de er positiv, negativ eller nøytral til elbiler"},
            {"role": "user", "content": f"Under er en redditpost sammen med kommentarfeltet i en liste av strenger - du skal vurdere om kommentarene eller person som skriver de er positiv, negativ eller nøytral/ikke-relevant til elbiler. 1 for positiv, -1 for negativ og 0 for nøytral, eller om artikkel/kommentar ikke er relevant\n{komm}"}
        ],
        response_format=sentimentanalyse
    )
    return completion


In [131]:

log = []

def analyser_post(df_post):
    
    dat = {"post": df_post["data.selftext"].iloc[0]}
    dat.update({str(i): kommentar for i,kommentar in enumerate(list(df_post["kommentarer"]))})

    resp = analyser(dat)
    data_resp = resp.choices[0].message.parsed.model_dump()
    log.append(data_resp)
    ny_df = pd.json_normalize(data_resp, "kommentar", meta="post").drop(columns="nummer")
    return ny_df


test = df.query("`data.title`.str.contains('Låne i frivæ')")
#test_dat = list(test["data.selftext"].unique())
#test_dat.extend(list(test["kommentarer"]))

#test_dat = {"post": test["data.selftext"].iloc[0]}
#kommentarer = {f"{i}": kommentar for i,kommentar in enumerate(list(test["kommentarer"]))}

#test_dat.update(kommentarer)

#resp = analyser(test_dat)

In [125]:
data = resp.choices[0].message.parsed.model_dump(

SyntaxError: incomplete input (418929359.py, line 1)

In [126]:
d = pd.json_normalize(data, "kommentar", meta="post").drop(columns="nummer")
tmp = pd.concat([d.set_index(test.index), test], axis=1)




In [113]:
df_test = df.query("`data.selftext`.str.contains('Jeg tog turen')")
test_res = df_test.groupby(by="data.id").apply(analyser_post, include_groups=False)


In [133]:
%%time
big = df.groupby(by="data.id").apply(analyser_post, include_groups=False)


KeyError: "['nummer'] not found in axis"

[{'post': 0,
  'kommentar': [{'nummer': 0, 'analyse': 0},
   {'nummer': 1, 'analyse': 0},
   {'nummer': 2, 'analyse': 0},
   {'nummer': 3, 'analyse': -1},
   {'nummer': 4, 'analyse': 0},
   {'nummer': 5, 'analyse': 0},
   {'nummer': 6, 'analyse': 0},
   {'nummer': 7, 'analyse': 0},
   {'nummer': 8, 'analyse': 1},
   {'nummer': 9, 'analyse': 1},
   {'nummer': 10, 'analyse': -1},
   {'nummer': 11, 'analyse': 0},
   {'nummer': 12, 'analyse': 0},
   {'nummer': 13, 'analyse': 0},
   {'nummer': 14, 'analyse': -1},
   {'nummer': 15, 'analyse': 1},
   {'nummer': 16, 'analyse': 1},
   {'nummer': 17, 'analyse': 0},
   {'nummer': 18, 'analyse': 0},
   {'nummer': 19, 'analyse': 0},
   {'nummer': 20, 'analyse': 0},
   {'nummer': 21, 'analyse': 0},
   {'nummer': 22, 'analyse': 1},
   {'nummer': 23, 'analyse': 0},
   {'nummer': 24, 'analyse': 0},
   {'nummer': 25, 'analyse': 0},
   {'nummer': 26, 'analyse': 0},
   {'nummer': 27, 'analyse': 0},
   {'nummer': 28, 'analyse': 0},
   {'nummer': 29, 'analy